In [ ]:
# Import
import pandas as pd 
# Read the production data CSV file into a DataFrame
clean_data = pd.read_csv("production_data.csv")
# Create columns for the DataFrame with the specified names
columns = ["batch_id",
          "production_date",
          "raw_material_supplier",
          "pigment_type",
          "pigment_quantity",
          "mixing_time",
          "mixing_speed",
          "product_quality_score"]
# Keep only the specified columns in the DataFrame and convert the production_date column to datetime format, dropping any rows with missing values in batch_id or production_date
clean_data = clean_data[columns].copy()
clean_data["production_date"] = pd.to_datetime(
    clean_data["production_date"], errors="coerce")
clean_data = clean_data.dropna(subset=["batch_id", "production_date"])
# Clean and standardize the raw_material_supplier column by converting it to lowercase, stripping whitespace, and mapping values to either "national_supplier" or "international_supplier". Any unmapped values are filled with "national_supplier".
supplier = clean_data["raw_material_supplier"].astype("string").str.strip().str.lower()
clean_data["raw_material_supplier"] = supplier.map({
    "1": "national_supplier",
    "1.0": "national_supplier",
    "2": "international_supplier",
    "2.0": "international_supplier",
    "national_supplier": "national_supplier",
    "international_supplier": "international_supplier",
}).fillna("national_supplier")
# Clean and standardize the pigment_type column by converting it to lowercase, stripping whitespace, and mapping values to either "type_a", "type_b", or "type_c". Any unmapped values are filled with "other".
pigment = clean_data["pigment_type"].astype("string").str.strip().str.lower()
clean_data["pigment_type"] = pigment.where(pigment.isin(["type_a", "type_b", "type_c"]), "other").fillna("other")
# Clean and standardize the pigment_quantity column by converting it to numeric, coercing errors to NaN, and replacing any values outside the range of 1 to 100 with NaN. Then, fill any remaining NaN values with the median of the column.
clean_data["pigment_quantity"] = pd.to_numeric(clean_data["pigment_quantity"], errors="coerce")
clean_data.loc[~clean_data["pigment_quantity"].between(1, 100), "pigment_quantity"] = pd.NA
clean_data["pigment_quantity"] = clean_data["pigment_quantity"].fillna(
    round(clean_data["pigment_quantity"].median(), 2)
)
# Clean and standardize the mixing_time column by converting it to numeric, coercing errors to NaN, and replacing any negative values with NaN. Then, fill any remaining NaN values with the mean of the column rounded to 2 decimal places.
clean_data["mixing_time"] = pd.to_numeric(clean_data["mixing_time"], errors="coerce")
clean_data.loc[clean_data["mixing_time"] < 0, "mixing_time"] = pd.NA
clean_data["mixing_time"] = clean_data["mixing_time"].fillna(
    round(clean_data["mixing_time"].mean(), 2)
)
# Clean and standardize the mixing_speed column by converting it to lowercase, stripping whitespace, and mapping values to either "Low", "Medium", or "High". Any unmapped values are filled with "Not Specified".
speed = clean_data["mixing_speed"].astype("string").str.strip().str.lower()
clean_data["mixing_speed"] = speed.map({
    "low": "Low",
    "medium": "Medium",
    "high": "High",
}).fillna("Not Specified")
# Clean and standardize the product_quality_score column by converting it to numeric, coercing errors to NaN, and replacing any values outside the range of 1 to 10 with NaN. Then, fill any remaining NaN values with the mean of the column rounded to 2 decimal places.
clean_data["product_quality_score"] = pd.to_numeric(
    clean_data["product_quality_score"], errors = "coerce"
)
clean_data.loc[~clean_data["product_quality_score"].between(1, 10), "product_quality_score"] = pd.NA
clean_data["product_quality_score"] = clean_data["product_quality_score"].fillna(
    round(clean_data["product_quality_score"].mean(), 2)
)

In [28]:
# Part 2
# Calculate the average `product_quality_score` and `pigment_quantity` grouped by `raw_material_supplier`.
aggregated_data = (
    clean_data.groupby("raw_material_supplier", as_index=False)
    .agg(
        avg_product_quality_score=("product_quality_score", "mean"),
        avg_pigment_quantity=("pigment_quantity", "mean"),
    )
    .round(2)
)

In [29]:
# Part 3
# Filter the data to include only rows where `raw_material_supplier` is 2 and `pigment_quantity` is greater than 35. Then, calculate the average `pigment_quantity` and `product_quality_score` grouped by `raw_material_supplier`.
import pandas as pd
# Read the production data CSV file into a DataFrame
production_data = pd.read_csv("production_data.csv")
# Create columns for the DataFrame with the specified names
filtered_data = production_data[
    (production_data["raw_material_supplier"] == 2)
    & (production_data["pigment_quantity"] > 35)
]
# Calculate the average `pigment_quantity` and `product_quality_score` grouped by `raw_material_supplier`.
pigment_data = (
    filtered_data.groupby("raw_material_supplier", as_index=False)
    .agg(
        pigment_quantity=("pigment_quantity", "mean"),
        avg_product_quality_score=("product_quality_score", "mean"),
    )
    .round(2)
)

In [30]:
# Part 4
# Calculate the mean, standard deviation, and correlation coefficient for `product_quality_score` and `pigment_quantity`.
import pandas as pd
# Read the production data CSV file into a DataFrame
production_data = pd.read_csv("production_data.csv")
# Create columns for the DataFrame with the specified names
for column in ["pigment_quantity", "product_quality_score"]:
    production_data[column] = pd.to_numeric(production_data[column], errors="coerce")
# Replace any values outside the valid ranges with NaN
production_data.loc[
    ~production_data["pigment_quantity"].between(1, 100), "pigment_quantity"
] = pd.NA
production_data.loc[
    ~production_data["product_quality_score"].between(1, 10),
    "product_quality_score",
] = pd.NA
# Fill any remaining NaN values with the mean of the respective columns
product_quality = pd.DataFrame([{
    "product_quality_score_mean": production_data["product_quality_score"].mean(),
    "product_quality_score_sd": production_data["product_quality_score"].std(),
    "pigment_quantity_mean": production_data["pigment_quantity"].mean(),
    "pigment_quantity_sd": production_data["pigment_quantity"].std(),
    "corr_coef": production_data["pigment_quantity"].corr(
        production_data["product_quality_score"]
    ),
}]).round(2)